# Bulk Download: Time Series for One Bank

Goal: pull Call Report XBRL data for one bank across several quarters, combine it into a single DataFrame, and plot a metric over time.

**Use case:** You're tracking how a bank's total assets, loans, or deposits have evolved quarter-over-quarter.

**Tip:** The library's built-in rate limiter (2500 req/hr) means you can safely pull dozens of quarters in sequence without worrying about throttling.

## Setup

In [ ]:
import os
import pandas as pd
from ffiec_data_connect import (
    OAuth2Credentials,
    collect_data,
    collect_reporting_periods,
)

creds = OAuth2Credentials(
    username=os.environ['FFIEC_USERNAME'],
    bearer_token=os.environ['FFIEC_BEARER_TOKEN'],
)

## Pick a bank and a date range

For this example: Bank of America (RSSD 480228), last 8 quarters.

In [ ]:
RSSD_ID = '480228'  # Bank of America, N.A.
NUM_QUARTERS = 8

all_periods = collect_reporting_periods(creds, series='call')
# Skip the latest period — it may be a future quarter not yet filed
target_periods = all_periods[-(NUM_QUARTERS + 1):-1]

print(f'Fetching {len(target_periods)} quarters:')
for p in target_periods:
    print(f'  {p}')

## Fetch each quarter

Each call returns the full XBRL filing. We tag each row with its reporting period so we can concatenate into a long-format DataFrame.

In [ ]:
quarterly_data = []

for period in target_periods:
    print(f'Fetching {period}...', end=' ')
    df = collect_data(
        creds,
        reporting_period=period,
        rssd_id=RSSD_ID,
        series='call',
        output_type='pandas',
    )
    df['reporting_period'] = period
    quarterly_data.append(df)
    print(f'{len(df)} items')

combined = pd.concat(quarterly_data, ignore_index=True)
print(f'\nCombined: {len(combined):,} rows')

## Extract one metric across time

Call Report data uses MDRM codes to identify individual data points. For example:
- `RCFD2170` — Total assets (consolidated)
- `RCFD2200` — Total deposits
- `RCFD2122` — Total loans and leases, net

See the [Call Report MDRM Data Dictionary](https://cdr.ffiec.gov/public/ManageFacsimiles.aspx) for the full list.

In [ ]:
# Extract total assets over time
MDRM_TOTAL_ASSETS = 'RCFD2170'

time_series = (
    combined[combined['mdrm'] == MDRM_TOTAL_ASSETS]
    [['reporting_period', 'int_data']]
    .rename(columns={'int_data': 'total_assets_thousands'})
    .sort_values('reporting_period')
    .reset_index(drop=True)
)

time_series

## Plot it

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(time_series['reporting_period'], time_series['total_assets_thousands'] / 1e6, marker='o')
ax.set_title(f'Total Assets Over Time — RSSD {RSSD_ID}')
ax.set_xlabel('Reporting Period')
ax.set_ylabel('Total Assets ($B)')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Save for later analysis

For production pipelines, save the combined DataFrame to Parquet — efficient, preserves dtypes, and fast to reload.

In [ ]:
combined.to_parquet('bank_480228_8q.parquet', index=False)
print(f'Saved {len(combined):,} rows')